# Lab: glacier and ice-sheet response

```{note}
This lab accompanies {doc}`glacier-variations`. Every equation here is derived in that chapter, and the lab's job is to make the reduced models run, wander, lag, and go unstable in front of you.
```

A glacier terminus is a low-pass filter applied to climate, and a marine ice stream is the same idea with a moving boundary that can run away. This lab builds the whole family of reduced models from {doc}`glacier-variations` in one place, from the single-reservoir mountain glacier to the two-reservoir marine ice stream, and forces each with steps, noise, ramps, and a deliberately retrograde bed. By the end you should be able to

- integrate the one-stage response equation and verify the 63% and 86% checkpoints of the exponential step response,
- chain three linear reservoirs into the Roe–Baker model and see why its sigmoidal onset is the realistic one,
- force the land models with white-noise mass balance and compare the red-noise wanderings they produce,
- run a warming ramp and measure the lag and the committed retreat for two real glaciers,
- integrate the Robel–Roe–Haseloff two-stage marine model, extract its two response timescales from the Jacobian, map the stability number across bed slopes, and watch a marine ice stream wander and go unstable under ocean noise.

Pure numpy and matplotlib throughout; the lab runs in seconds, which is the whole point of a reduced model.

---

## Part I — the one-stage land glacier

The chapter reduced a glacier to a single reservoir. A persistent mass-balance perturbation $\dot b'$ fills it, ablation over the advancing terminus wedge drains it, and the length perturbation $L'$ obeys

$$ \frac{dL'}{dt} + \frac{L'}{\tau} = \dot b'\,\frac{\bar Y}{Y_t}\,\frac{L_0}{H_0}, \qquad \tau = \frac{H}{|\dot b_t|}, $$

with $\tau$ the Jóhannesson-Raymond-Waddington response time, the ice thickness over the (magnitude of the) terminus balance rate. The chapter wrote this timescale $t_r$; we switch to $\tau$ here to match the Roe-Baker notation that arrives below, and we set the width ratio $\bar Y / Y_t = 1$ throughout so the geometry factor reduces to $L_0/H_0$. For a step change in balance the solution is exponential, reaching 63% of the new equilibrium after one $\tau$ and 86% after two.

Two real glaciers from the chapter will follow us through the whole lab.

| Glacier | $H$ (m) | $\dot b_t$ (m yr$^{-1}$) | $\tau$ (yr) | $L_0$ (km) |
|---|---|---|---|---|
| Nisqually (Mount Rainier) | 96 | $-8.0$ | 12 | 7.0 |
| South Cascade | 200 | $-5.4$ | 37 | 3.4 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

glaciers = {
    'Nisqually':     dict(H=96.0,  bt=-8.0, L0=7000.0),
    'South Cascade': dict(H=200.0, bt=-5.4, L0=3400.0),
}

for name, g in glaciers.items():
    g['tau'] = g['H'] / abs(g['bt'])
    g['geom'] = g['L0'] / g['H']      # the factor (Ybar/Yt)(L0/H0) with Ybar/Yt = 1
    print(f"{name:14s}  tau = {g['tau']:5.1f} yr   L0/H0 = {g['geom']:5.1f}")

**Task 1:** Complete the forward-Euler integrator below, which takes an arbitrary time series of balance perturbations. Apply a step of $\dot b' = +0.5\ \mathrm{m\,yr^{-1}}$ switched on at $t = 0$ to both glaciers, plot $L'(t)/\Delta L_{\mathrm{eq}}$ against $t/\tau$, and confirm that the two curves collapse onto a single exponential. Print the fraction of the equilibrium response reached at $t = \tau$ and $t = 2\tau$, which should land near 0.63 and 0.86. The equilibrium response itself is $\Delta L_{\mathrm{eq}} = \tau\,\dot b'\,L_0/H_0$; compute it for each glacier and notice how a half-meter balance change moves a terminus by hundreds of meters.

In [ ]:
def one_stage(t, bdot, tau, geom):
    """Integrate dL'/dt + L'/tau = bdot(t) * geom with forward Euler.

    t    : uniform time array, yr
    bdot : array of balance perturbations at the times t, m/yr
    tau  : response time, yr
    geom : the geometric factor L0/H0 (dimensionless)
    """
    L = np.zeros_like(t)
    dt = t[1] - t[0]
    for i in range(len(t) - 1):
        # YOUR CODE HERE: update L[i+1] from L[i]
        pass
    return L


dt = 0.1
t = np.arange(0.0, 200.0 + dt, dt)
bdot_step = np.full_like(t, 0.5)

fig, ax = plt.subplots(figsize=(6, 4))
for name, g in glaciers.items():
    # L = one_stage(t, bdot_step, g['tau'], g['geom'])
    # dL_eq = g['tau'] * 0.5 * g['geom']
    # ax.plot(t / g['tau'], L / dL_eq, lw=2, label=name)
    # YOUR CODE HERE: print dL_eq and the fractions reached at tau and 2*tau
    pass
ax.axhline(1 - np.exp(-1), color='gray', ls=':')
ax.axhline(1 - np.exp(-2), color='gray', ls=':')
ax.set_xlabel('t / tau')
ax.set_ylabel('fraction of equilibrium response')
ax.legend()
plt.show()

## Part II — the three-stage land glacier

The one-stage model has a flaw visible in your Task 1 plot. Its terminus starts moving at $t = 0$ at its maximum rate, as if a balance perturbation falling on the accumulation zone could move the terminus the same year. A real glacier responds in sequence. The interior thickens first, the extra thickness drives extra flux toward the terminus, and only when that flux arrives does the length change in earnest. Roe and Baker chained three linear reservoirs to capture exactly this,

$$ \frac{dh'}{dt}+\frac{h'}{\epsilon\tau}=\dot b', \qquad \frac{dF'}{dt}+\frac{F'}{\epsilon\tau}=\frac{L_0\,h'}{(\epsilon\tau)^{2}}, \qquad \frac{dL'}{dt}+\frac{L'}{\epsilon\tau}=\frac{F'}{\epsilon H_0}, $$

with the same $\tau$ as before and $\epsilon = 1/\sqrt{3}$ chosen so that the chain preserves the equilibrium response and the overall memory of the one-stage model. The step response, derived in the chapter, is sigmoidal,

$$ L'(t) = L'_{\mathrm{eq}}\left[1 - e^{-t/\epsilon\tau}\left(1 + \frac{t}{\epsilon\tau} + \frac{1}{2}\left(\frac{t}{\epsilon\tau}\right)^{2}\right)\right], $$

flat at first while the anomaly works its way down-glacier, then catching up.

**Task 2:** Complete `three_stage` below, integrating the three equations with forward Euler. Apply the same $+0.5\ \mathrm{m\,yr^{-1}}$ step to South Cascade, overlay the one-stage response, your numerical three-stage response, and the analytic sigmoid, and check that all reach the same equilibrium. Print the fraction of equilibrium reached at $t = \tau, 2\tau, 3\tau$ for both models. Which model would you rather have fit an e-folding time to?

In [ ]:
EPS = 1.0 / np.sqrt(3.0)


def three_stage(t, bdot, tau, L0, H0):
    """Roe-Baker three-stage model, forward Euler.

    Returns the length perturbation L' (m) at the times t (yr).
    State variables: h (interior thickness anomaly, m),
    F (terminus flux anomaly, m^2/yr), L (length anomaly, m).
    """
    h = np.zeros_like(t)
    F = np.zeros_like(t)
    L = np.zeros_like(t)
    dt = t[1] - t[0]
    et = EPS * tau
    for i in range(len(t) - 1):
        # YOUR CODE HERE: update h[i+1], F[i+1], L[i+1] from the
        # three chained equations
        pass
    return L


g = glaciers['South Cascade']
tau, L0, H0 = g['tau'], g['L0'], g['H']
dL_eq = tau * 0.5 * L0 / H0

et = EPS * tau
x = t / et
L_analytic = dL_eq * (1 - np.exp(-x) * (1 + x + 0.5 * x**2))

fig, ax = plt.subplots(figsize=(6, 4))
# L1 = one_stage(t, bdot_step, tau, L0 / H0)
# L3 = three_stage(t, bdot_step, tau, L0, H0)
# ax.plot(t, L1, lw=2, label='one stage')
# ax.plot(t, L3, lw=2, label='three stage (numerical)')
ax.plot(t, L_analytic, 'k--', lw=1, label='three stage (analytic)')
ax.set_xlabel('time (yr)')
ax.set_ylabel("L' (m)")
ax.legend()
plt.show()

# YOUR CODE HERE: print the fraction of dL_eq reached at tau, 2*tau,
# 3*tau for both models

## Part III — stochastic forcing of land glaciers

Interannual mass-balance variability is essentially white noise, a meter or so of water equivalent from year to year with little memory. The glacier integrates it. Run white noise through either model and the terminus performs slow, red-noise wanderings over decades, with no climate trend anywhere in the forcing. For the one-stage model forced annually ($\Delta t = 1$ yr) with noise of standard deviation $\sigma_b$, Roe and Baker give the closed-form variance

$$ \sigma_L^2 = \frac{\tau\,\Delta t}{2}\left(\frac{L_0}{H_0}\right)^{2}\sigma_b^2, $$

and the three-stage model, which filters high frequencies more aggressively, wanders with roughly three quarters of that variance for glaciers like these.

**Task 3:** Generate 10,000 years of annual balance anomalies drawn from $N(0, \sigma_b^2)$ with $\sigma_b = 1.0\ \mathrm{m\,yr^{-1}}$, run both models for South Cascade with $\Delta t = 1$ yr, and plot a 2,000-year window of the two length records. Compute the standard deviation of each, compare the one-stage value with the analytic prediction, and report the three-stage to one-stage variance ratio. Then answer the question this experiment exists to pose. If you were handed a single length record showing a 300-m advance over 50 years, could you conclude the climate had changed?

In [ ]:
rng = np.random.default_rng(seed=431)

dt_n = 1.0
t_n = np.arange(0.0, 10000.0, dt_n)
sigma_b = 1.0
bdot_noise = rng.normal(0.0, sigma_b, size=t_n.size)

g = glaciers['South Cascade']

# L1 = one_stage(t_n, bdot_noise, g['tau'], g['geom'])
# L3 = three_stage(t_n, bdot_noise, g['tau'], g['L0'], g['H'])

fig, ax = plt.subplots(figsize=(9, 4))
window = slice(4000, 6000)
# ax.plot(t_n[window], L1[window], lw=1, label='one stage')
# ax.plot(t_n[window], L3[window], lw=1.5, label='three stage')
ax.axhline(0.0, color='gray', lw=0.5)
ax.set_xlabel('time (yr)')
ax.set_ylabel("L' (m)")
ax.legend()
plt.show()

sigma_L_analytic = np.sqrt(g['tau'] * dt_n / 2.0) * g['geom'] * sigma_b
print(f'analytic one-stage sigma_L = {sigma_L_analytic:.0f} m')
# YOUR CODE HERE: print the numerical sigma_L for both models (discard
# the first ~500 yr of spin-up) and the three-stage/one-stage variance ratio

## Part IV — ramp forcing and committed retreat

Now force the models with something resembling the last century. A warming trend $T'(t) = \dot T t$ maps onto mass balance through the melt factor $\mu$, so $\dot b'(t) = -\mu\,T'(t)$, and we take $\mu = 0.65\ \mathrm{m\,yr^{-1}\,K^{-1}}$ from Roe and Baker. Apply $\dot T = 0.02\ \mathrm{K\,yr^{-1}}$ for 100 years, then hold the climate fixed at its new, 2-degree-warmer state for another 150 years.

At every instant there is an equilibrium length the glacier would have if it could keep up, $L'_{\mathrm{eq}}(t) = \tau\,\dot b'(t)\,L_0/H_0$. During the ramp the terminus chases this target and falls behind it by an amount that grows with $\tau$, and when the warming stops the chase continues. The gap between actual and equilibrium length at the moment the climate stabilizes is the **committed retreat**, the retreat already purchased but not yet delivered.

**Task 4:** Run the three-stage model through the ramp-and-hold scenario for both glaciers. For each, plot $L'(t)$ with $L'_{\mathrm{eq}}(t)$ dashed, and print the committed retreat at year 100. Then answer two questions from the plot. Which glacier tracks its climate, and which carries a bank of unrealized retreat? And how many years after the warming stops does each glacier take to close 90% of its gap?

In [ ]:
mu = 0.65        # melt factor, m/yr per K
Tdot = 0.02      # warming rate, K/yr
t_ramp = 100.0   # duration of the ramp, yr

dt_r = 0.1
t_r = np.arange(0.0, 250.0 + dt_r, dt_r)
Tprime = np.where(t_r < t_ramp, Tdot * t_r, Tdot * t_ramp)
bdot_ramp = -mu * Tprime

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, (name, g) in zip(axes, glaciers.items()):
    # L = three_stage(t_r, bdot_ramp, g['tau'], g['L0'], g['H'])
    # L_eq = g['tau'] * bdot_ramp * g['geom']
    # ax.plot(t_r, L, lw=2, label='terminus')
    # ax.plot(t_r, L_eq, 'k--', lw=1, label='equilibrium')
    # YOUR CODE HERE: committed retreat = L_eq - L at t = 100 yr; print it,
    # and find when the remaining gap first falls below 10% of that value
    ax.axvline(t_ramp, color='gray', ls=':')
    ax.set_title(name)
    ax.set_xlabel('time (yr)')
    ax.set_ylabel("L' (m)")
    ax.legend()
plt.tight_layout()
plt.show()

---

## Part V — the two-stage marine ice stream

For a marine-terminating glacier the terminus wedge is the wrong picture, and {doc}`glacier-variations` replaces it with two reservoirs, the interior thickness $H$ and the grounding-line position $L$,

$$
\frac{dL}{dt} = \frac{Q - Q_g}{h_g}, \qquad
\frac{dH}{dt} = P - \frac{Q_g}{L} - \frac{H}{h_g L}\left(Q - Q_g\right),
$$

with interior flux $Q = \nu H^{\alpha}/L^{\gamma}$, grounding-zone flux $Q_g = \Omega h_g^{\beta}$, and the flotation thickness pinned to the bed, $h_g = -\lambda b(L)$. We set up the model by choosing the steady state first, a grounding line at 400 km on a prograde bed with accumulation 0.3 m/yr, and deriving the flux prefactors $\nu$ and $\Omega$ from the steady-state balance $PL = Q = Q_g$, which guarantees the model starts in equilibrium. The integrator is RK4, since the two-stage system is stiff near its slow eigenvalue and forward Euler would need tiny steps.

In [ ]:
# --- two-stage marine model ---
lam = 1.1          # rho_w / rho_i
n_glen, m_fric = 3.0, 1.0/3.0
alpha, gamma = 2*n_glen + 2, n_glen          # interior-flux exponents (SIA)
beta = (m_fric + n_glen + 3) / (m_fric + 1)  # ~4.75, Schoof grounding-line exponent

def make_bed(b0=-100.0, bx=-1.0e-3):
    """Linear bed elevation b(x) = b0 + bx*x  (bx<0: deepens seaward, prograde)."""
    return lambda x: b0 + bx*x

bed = make_bed()

# Choose the steady state, then derive the prefactors
P = 0.3            # m/yr
L0m, H0m = 400e3, 1500.0
hg0 = -lam * bed(L0m)
Omega = P * L0m / hg0**beta
nu = P * L0m**(gamma + 1) / H0m**alpha

def fluxes(H, L, Om, bedfn):
    hg = -lam * bedfn(L)
    Q = nu * H**alpha / L**gamma
    Qg = Om * hg**beta
    return Q, Qg, hg

def rhs(state, Om, bedfn):
    H, L = state
    Q, Qg, hg = fluxes(H, L, Om, bedfn)
    dL = (Q - Qg) / hg
    dH = P - Qg / L - H * (Q - Qg) / (hg * L)
    return np.array([dH, dL])

def rk4(state, Om, dt, bedfn):
    k1 = rhs(state, Om, bedfn)
    k2 = rhs(state + 0.5*dt*k1, Om, bedfn)
    k3 = rhs(state + 0.5*dt*k2, Om, bedfn)
    k4 = rhs(state + dt*k3, Om, bedfn)
    return state + dt*(k1 + 2*k2 + 2*k3 + k4)/6

Q, Qg, hg = fluxes(H0m, L0m, Omega, bed)
print(f"steady-state check: PL = {P*L0m:.4g}, Q = {Q:.4g}, Qg = {Qg:.4g} m^2/yr")

**Task 5: step response and the two timescales.** Increase $\Omega$ by twenty percent at $t=0$, the reduced-model version of an ocean-melt forcing, and integrate for five thousand years. Plot $H(t)$ and $L(t)$. Then compute the eigenvalues of the numerical Jacobian at the original steady state and compare their inverse magnitudes against the fast and slow phases you see in the trajectories.

In [ ]:
dt = 0.5
nsteps = int(5000 / dt)
state = np.array([H0m, L0m])
traj = np.empty((nsteps, 2))
for i in range(nsteps):
    state = rk4(state, 1.2 * Omega, dt, bed)
    traj[i] = state
tm = dt * np.arange(1, nsteps + 1)

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(tm, traj[:, 0]); axes[0].set_ylabel("H (m)")
axes[1].plot(tm, traj[:, 1] / 1e3); axes[1].set_ylabel("L (km)")
axes[1].set_xlabel("time (yr)")
fig.suptitle("Two-stage step response to a 20% increase in $\\Omega$")

def jacobian(state, Om, bedfn, eps=1e-4):
    J = np.zeros((2, 2)); f0 = rhs(state, Om, bedfn)
    for k in range(2):
        ds = state.copy(); ds[k] *= (1 + eps)
        J[:, k] = (rhs(ds, Om, bedfn) - f0) / (state[k] * eps)
    return J

ev = np.linalg.eigvals(jacobian(np.array([H0m, L0m]), Omega, bed))
print("eigenvalues (1/yr):", ev)
print("e-folding times (yr):", np.sort(-1 / ev.real))

## Part VI — the stability number and critical slowing

**Task 6.** Sweep the bed slope from strongly prograde ($b_x = -2\times10^{-3}$) toward the critical value, recomputing the steady state and the stability number

$$
\Sigma = \beta\,\frac{\bar L\, h_g'(\bar L)}{\bar h_g} - 1, \qquad h_g' = -\lambda\, b_x,
$$

at each slope. Plot the response gain $1/\Sigma$ and the slow e-folding time against $b_x$. Both diverge as $\Sigma \to 0$, the critical slowing derived in the chapter. What does the divergence imply for the trustworthiness of a sensitivity estimate made near a flat bed?

In [ ]:
slopes = np.linspace(-2.0e-3, -0.05e-3, 40)
Sig, tau_slow = [], []
for bx in slopes:
    bedfn = make_bed(bx=bx)
    hg_s = -lam * bedfn(L0m)
    Om_s = P * L0m / hg_s**beta           # re-derive Omega so (H0m, L0m) stays steady
    Sigma = beta * L0m * (-lam * bx) / hg_s - 1
    ev = np.linalg.eigvals(jacobian(np.array([H0m, L0m]), Om_s, bedfn))
    Sig.append(Sigma); tau_slow.append(np.max(-1 / ev.real))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(slopes * 1e3, 1 / np.array(Sig))
axes[0].set_xlabel("bed slope (m/km)"); axes[0].set_ylabel("gain $1/\\Sigma$")
axes[1].semilogy(slopes * 1e3, tau_slow)
axes[1].set_xlabel("bed slope (m/km)"); axes[1].set_ylabel("slow e-folding time (yr)");

## Part VII — fifty thousand years of ocean noise

The experiment the marine model exists for. Hold the climate statistically steady, $\Omega(t) = \bar\Omega\,(1 + \sigma\,\xi_t)$ with $\xi_t$ independent standard-normal draws each year and $\sigma = 0.2$, a modest ocean variability with no trend whatever. Integrate for 50,000 years on the prograde bed, plot $L(t)$, and compute its standard deviation and decorrelation time.

**Task 7.** Run the simulation below, then answer in writing, with reference to the red-noise filtering and the stability discussion of {doc}`glacier-variations`:

1. How large are the grounding-line excursions, and over what timescales do they unfold, given that the forcing has zero memory from year to year? Compare with the land-glacier wanderings of Part III.
2. A satellite record shows fifteen years of steady grounding-line retreat at a glacier on a prograde bed. Using your simulated null distribution, how strong is that evidence of forced change?
3. Restart the run with the grounding line just upstream of a retrograde reach (build a bed with `make_bed(bx=+5e-4)` over the relevant interval, or flip the slope sign locally). How does noise interact with the instability threshold, and what does that mean for the deep-uncertainty tail of {doc}`sea-level`?

In [ ]:
rng = np.random.default_rng(7)
years = 50_000
dt = 0.5
steps_per_year = int(1 / dt)
sigma = 0.2

state = np.array([H0m, L0m])
L_series = np.empty(years)
for yr in range(years):
    Om_t = Omega * (1 + sigma * rng.standard_normal())
    for _ in range(steps_per_year):
        state = rk4(state, max(Om_t, 0.1 * Omega), dt, bed)
    L_series[yr] = state[1]

fig, axes = plt.subplots(figsize=(9, 3.5))
axes.plot(np.arange(years) / 1e3, (L_series - L0m) / 1e3, lw=0.6)
axes.set_xlabel("time (kyr)"); axes.set_ylabel("grounding-line anomaly (km)")
print(f"std of L: {np.std(L_series)/1e3:.2f} km")
ac = np.correlate(L_series - L_series.mean(), L_series - L_series.mean(), "full")[years-1:]
ac /= ac[0]
print(f"decorrelation time (1/e): {np.argmax(ac < 1/np.e)} yr")

---

## Synthesis

Draw the whole family of models together in a few short paragraphs.

1. The response time $\tau$ is an e-folding scale, not a delay, and the three-stage model of Part II shows the difference. Explain why fitting a simple exponential to an observed terminus record would overestimate a glacier's memory, and why the distinction matters when an observed retreat is tested against the null hypothesis of noise-driven wandering.
2. Your Part III wanderings and your Part IV committed retreat are two faces of the same land-glacier filter. State, in one or two sentences each, what each implies about how a single glacier length record should and should not be read.
3. The two-stage marine model of Parts V–VII produced the marine ice-sheet instability from nothing but mass conservation and a flux rule. What does that tell you about what the instability fundamentally is? And why does the millennial slow timescale mean that a marine ice stream which has looked stable for centuries may be telling you nothing of the kind?
4. Set this lab beside the icepack experiment of {doc}`../modeling/ice-stream-vaf-lab`. The full stress-balance model and the reduced model agree on the phenomenology, fast partial adjustment, slow completion, runaway on retrograde slopes. State the one question you would put to each tool rather than the other, and the one ingredient of the full model that the two-stage model cannot represent at all.